In [0]:
# 1. Lendo arquivos parquet
df_novos = spark.read.parquet("/Volumes/projeto_bcb/bronze/arquivos_brutos/cotacoes_novos/*")

# 2. Limpeza de dados 
df_limpo = spark.sql("""
    SELECT 
        cotacaoCompra AS Cotacao,
        CAST(dataHoraCotacao AS DATE) AS Data,
        moeda AS Moeda
    FROM {df_novos}
    WHERE moeda IS NOT NULL
""", df_novos=df_novos).dropDuplicates(["Moeda", "Data"])

# 3. Criando a View
df_limpo.createOrReplaceTempView("vw_cotacoes_novas")

display(df_limpo)

In [0]:
%sql

DROP TABLE IF EXISTS projeto_bcb.silver.cotacoes;

-- Cria a tabela
CREATE TABLE projeto_bcb.silver.cotacoes (
    Cotacao DOUBLE,
    Data DATE,
    Moeda STRING
)
USING DELTA
PARTITIONED BY (Moeda);

-- Executa o MERGE (Upsert)
MERGE INTO projeto_bcb.silver.cotacoes AS destino
USING vw_cotacoes_novas AS origem
ON destino.Moeda = origem.Moeda AND destino.Data = origem.Data
WHEN NOT MATCHED THEN
    INSERT (Cotacao, Data, Moeda)
    VALUES (origem.Cotacao, origem.Data, origem.Moeda);